# Voice2Reel 07 — استوديو Stickman كامل: تدريب مرة واحدة ← محتوى ← فيديو صامت ← صوت ← دمج

**اه، المنطق اللي قلته صح:** ما بندرّب الموديل كل ما بدنا ننشر فيديو. المفروض يطلع **ملف واحد بنتائج التدريب**، وكل الخلايا اللي بعده تقرأ هذا الملف تلقائياً.

هذا الدفتر يسوي بالترتيب:

0. تجهيز الموارد + فحص البيانات.
1. **تدريب مرة واحدة**: `Qwen2.5-1.5B` LoRA adapter + مولّد نقاط Stickman MLP + كاشف عيوب + بوابة CPU، ثم حفظ `model_training_results.json` ونسخة `نتائج_تدريب_الموديل.json`.
2. الخلايا التالية **تحمل الملف ولا تعيد التدريب** إلا إذا عملت `FORCE_RETRAIN = True`.
3. تكتب **عنوان الفيديو + الفكرة + النص المرجعي** في أول خلية؛ Qwen يولّد السكربت كامل والنص المنطوق لكل مشهد.
4. إنشاء **فيديو بدون صوت** من نقاط المولّد المحفوظ.
5. إنشاء **الصوت** لكل مشهد.
6. إعادة ضبط توقيت الإطارات حسب الصوت الحقيقي ثم **دمج الصوت والفيديو**.
7. تقرير، عرض، وتنزيل.

حدود مهمة: هذا خط إنتاج تجريبي كامل (نص + صوت + صورة متجهية + فيديو)، لكنه **ليس** مولّد صور diffusion، ولا مراجعة بشرية، ولا `final_report.json` الخاص بكاتب الإنتاج `Qwen3-4B`. `edge-tts` خدمة خارجية وغير مُرخّصة تجارياً هنا. يحتاج **T4 GPU** للتدريب الأول فقط؛ بعدها نفس الملف يُستخدم للتوليد.

In [ ]:
# ===== الخلية الوحيدة اللي تعدّلها للمحتوى =====
VIDEO_REQUESTS = [
    {
        "title": "ليش الكود فجأة ظهر أحمر؟",
        "idea": "شرح سبب ظهور رسالة خطأ حمراء بعد تعديل بسيط على الكود",
        "source_text": "بعد تعديل بسيط يظهر خطأ أحمر في المحرر أو الطرفية. ابدأ بقراءة رسالة الخطأ، ثم ارجع لآخر تغيير، ثم جرّب إصلاحاً واحداً وتحقق من النتيجة.",
    },
    {
        "title": "كيف أتعامل مع المشاكل التقنية؟",
        "idea": "خطوات عملية لتشخيص أي مشكلة تقنية بدون تخمين",
        "source_text": "أعد إنتاج المشكلة أولاً. اقرأ رسالة الخطأ. غيّر شيئاً واحداً فقط. سجّل ما تغيّر. كرر حتى تعرف السبب.",
    },
]
GENERATE_ALL_IDEAS = False          # True = فيديو لكل فكرة في القائمة
SELECTED_IDEA_INDEX = 0             # استخدم 0 أو 1 عند GENERATE_ALL_IDEAS = False

DURATION = 30                       # 10–60 ثانية
LANGUAGE = "ar"
FPS = 12
VOICE = "ar-SA-ZariyahNeural"       # edge-tts تجريبي فقط
USE_EDGE_TTS = True                 # False = صوت صامت
FORCE_RETRAIN = False               # True = تجاهل ملف النتائج ودرّب من جديد

# إعدادات تدريب سريعة قابلة للزيادة على T4
QWEN_EPOCHS = 1
QWEN_MAX_STEPS = 80
MLP_EPOCHS = 30
print("Requests:", len(VIDEO_REQUESTS), "| selected:", SELECTED_IDEA_INDEX, "| duration:", DURATION)

## 0) تجهيز الموارد

Colab: `Runtime → Change runtime type → T4 GPU`. نثبّت حزم النص/الرسم/الصوت/الفيديو، ونفحص كل البيانات عبر `scripts/check_data.py`. لا نثبّت `torch` من جديد إن كان Colab يوفّره.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import re
import shutil
import subprocess
import sys

REPO = Path('/content/Voice2Reel')
if not (REPO / 'pyproject.toml').is_file():
    REPO = Path.cwd()
    while REPO != REPO.parent and not (REPO / 'pyproject.toml').is_file():
        REPO = REPO.parent
assert (REPO / 'pyproject.toml').is_file(), 'شغّل من جذر المستودع أو ارفع ZIP المستودع'
sys.path.insert(0, str(REPO))

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', '.',
     'transformers==4.57.3', 'peft==0.17.1', 'accelerate==1.10.1',
     'bitsandbytes==0.48.2', 'huggingface-hub==0.36.0',
     'Pillow==13.0.0', 'arabic-reshaper==3.1.0',
     'imageio-ffmpeg==0.6.0', 'edge-tts==7.3.1', 'nest_asyncio==1.6.0'],
    cwd=REPO, check=True,
)
try:
    import torch
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch'], cwd=REPO, check=True)
    import torch
import numpy as np
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import imageio_ffmpeg

if not torch.cuda.is_available():
    raise RuntimeError('اختر T4 GPU في Colab: تدريب Qwen يحتاج CUDA')
DEVICE = torch.device('cuda')
FFMPEG = imageio_ffmpeg.get_ffmpeg_exe()

WORK = REPO / 'outputs' / 'stickman_studio'
VIDEOS_DIR = WORK / 'videos'
ADAPTER_DIR = WORK / 'qwen_adapter'
MLP_PATH = WORK / 'stickman_mlp.pt'
TRAINING_RESULTS_PATH = WORK / 'model_training_results.json'
TRAINING_RESULTS_AR_PATH = WORK / 'نتائج_تدريب_الموديل.json'
for folder in (WORK, VIDEOS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

subprocess.run([sys.executable, 'scripts/check_data.py'], cwd=REPO, check=True)
subprocess.run([sys.executable, 'scripts/audit_training_data.py'], cwd=REPO, check=True)
print('torch', torch.__version__, '| device', DEVICE, '| ffmpeg', FFMPEG)
print('training results file:', TRAINING_RESULTS_PATH)

## 1) تدريب مرة واحدة وحفظ «نتائج تدريب الموديل»

الخلية التالية تعرّف دوال التدريب. اللي بعدها تقرر: إن كان الملف موجوداً و`FORCE_RETRAIN = False` تحمّله فوراً؛ وإلا تدرّب وتكتب الملف. الملف يشير إلى adapter Qwen وأوزان MLP وقواعد كاشف العيوب ونتائج القياس.

In [ ]:
from voice2reel.scriptwriter.train import MODEL_ID, MODEL_REVISION
from voice2reel.stickman_artist import (
    DefectDetector, StickmanArtist, is_professional, rows, run_rounds, vocabulary,
)

CANVAS = np.array([540.0, 960.0], dtype=np.float32)


def train_qwen_adapter():
    if ADAPTER_DIR.exists():
        shutil.rmtree(ADAPTER_DIR)
    ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
    command = [
        sys.executable, '-m', 'voice2reel.scriptwriter.train',
        '--output', str(ADAPTER_DIR),
        '--epochs', str(QWEN_EPOCHS),
        '--max-steps', str(QWEN_MAX_STEPS),
    ]
    subprocess.run(command, cwd=REPO, check=True)
    adapter = ADAPTER_DIR / 'adapter'
    if not (adapter / 'adapter_model.safetensors').is_file():
        raise RuntimeError('Qwen adapter weights were not saved')
    metrics = json.loads((ADAPTER_DIR / 'metrics.json').read_text(encoding='utf-8'))
    provenance = json.loads((adapter / 'provenance.json').read_text(encoding='utf-8'))
    return {
        'adapter_dir': str(adapter), 'run_dir': str(ADAPTER_DIR),
        'base_model': MODEL_ID, 'revision': MODEL_REVISION,
        'train_rows': provenance['train_rows'], 'metrics': metrics,
        'gpu': provenance['gpu'],
    }


def train_stickman_mlp():
    designer = StickmanArtist()
    train_prof = [row for row in rows('train') if is_professional(row)]
    validation_prof = [row for row in rows('validation') if is_professional(row)]
    test_prof = [row for row in rows('test') if is_professional(row)]
    x = np.vstack([designer.design(row) for row in train_prof]).astype(np.float32)
    y = np.vstack([
        np.asarray(row['points'], dtype=np.float32).reshape(-1, 2) / CANVAS
        for row in train_prof
    ]).reshape(len(train_prof), -1).astype(np.float32)
    mean = x.mean(axis=0)
    std = np.where(x.std(axis=0) < 1e-6, 1.0, x.std(axis=0)).astype(np.float32)
    x = ((x - mean) / std).astype(np.float32)
    torch.manual_seed(20261110)
    np.random.seed(20261110)
    model = nn.Sequential(
        nn.Linear(x.shape[1], 128), nn.ReLU(),
        nn.Linear(128, 128), nn.ReLU(),
        nn.Linear(128, y.shape[1]),
    ).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()
    loader = DataLoader(TensorDataset(torch.from_numpy(x), torch.from_numpy(y)),
                        batch_size=256, shuffle=True)
    history = []
    for _epoch in range(1, MLP_EPOCHS + 1):
        model.train()
        total = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
            total += float(loss.item()) * len(xb)
        history.append(round(total / len(loader.dataset), 6))

    def predict(row):
        design = ((designer.design(row) - mean) / std).astype(np.float32)
        with torch.no_grad():
            model.eval()
            output = model(torch.from_numpy(design).unsqueeze(0).to(DEVICE)).cpu().numpy()
        return np.clip(output.reshape(18, 2) * CANVAS, [0.0, 0.0], CANVAS)

    def mean_error(part, predictor):
        errors = []
        for row in part:
            predicted = np.asarray(predictor(row), dtype=np.float64).reshape(-1, 2)
            reference = np.asarray(row['points'], dtype=np.float64).reshape(-1, 2)
            errors.append(float(np.mean(np.linalg.norm(predicted - reference, axis=1))))
        return round(float(np.mean(errors)), 3)

    # Evaluate model choices on validation, never the held-out test set.
    eval_rows = validation_prof[:1000]
    mlp_val_error = mean_error(eval_rows, predict)
    ridge = StickmanArtist().fit(train_prof)
    ridge_val_error = mean_error(eval_rows, ridge.predict)
    # Test is reporting-only, not a model-selection input.
    test_rows = test_prof[:1000]
    mlp_error = mean_error(test_rows, predict)
    ridge_error = mean_error(test_rows, ridge.predict)
    torch.save({
        'state_dict': model.state_dict(), 'mean': mean.tolist(), 'std': std.tolist(),
        'input_dim': int(x.shape[1]), 'output_dim': int(y.shape[1]),
        'canvas': CANVAS.tolist(), 'epochs': MLP_EPOCHS,
    }, MLP_PATH)
    return {
        'weights': str(MLP_PATH), 'input_dim': int(x.shape[1]),
        'output_dim': int(y.shape[1]), 'epochs': MLP_EPOCHS,
        'train_professional_rows': len(train_prof),
        'final_mse_loss': history[-1],
        'mlp_validation_joint_error_px': mlp_val_error,
        'ridge_validation_joint_error_px': ridge_val_error,
        'mlp_mean_joint_error_px': mlp_error,
        'ridge_mean_joint_error_px': ridge_error,
    }


def train_defect_and_gate():
    detector = DefectDetector().fit(rows('train'))
    cpu = run_rounds(max_rounds=3)
    return {
        'rules': detector.rules,
        'test_accuracy': detector.accuracy(rows('test')),
        'cpu_validation_scores': [item['score'] for item in cpu['rounds']],
        'cpu_final_test_score': cpu['final_test_on_best_round']['score'],
        'cpu_integrity_checks': len(cpu['integrity']),
        'cpu_integrity_all_true': all(cpu['integrity'].values()),
    }


def training_ready(results):
    adapter = Path(results['qwen_adapter']['adapter_dir'])
    weights = Path(results['stickman_mlp']['weights'])
    return (adapter / 'adapter_model.safetensors').is_file() and weights.is_file()

print('Training helpers ready | keypoints:', len(vocabulary()['keypoint_order']))

In [ ]:
if TRAINING_RESULTS_PATH.is_file() and not FORCE_RETRAIN:
    TRAINING_RESULTS = json.loads(TRAINING_RESULTS_PATH.read_text(encoding='utf-8'))
    if not training_ready(TRAINING_RESULTS):
        raise RuntimeError('ملف النتائج ناقص: اجعل FORCE_RETRAIN = True وأعد التدريب')
    print('✅ تم تحميل نتائج التدريب — بدون إعادة تدريب:', TRAINING_RESULTS_PATH)
else:
    print('🚀 تدريب مرة واحدة: Qwen adapter + Stickman MLP + كاشف عيوب + بوابة CPU ...')
    qwen = train_qwen_adapter()
    mlp = train_stickman_mlp()
    gate = train_defect_and_gate()
    TRAINING_RESULTS = {
        'name': 'model_training_results',
        'name_ar': 'نتائج تدريب الموديل',
        'version': 'stickman-studio-v1',
        'trained_at': datetime.now(timezone.utc).isoformat(),
        'reuse_in_later_cells': True,
        'retrain_per_video': False,
        'qwen_adapter': qwen,
        'stickman_mlp': mlp,
        'defect_detector': gate,
        'dataset': {
            'total_rows': 60000,
            'splits': {'train': 50000, 'validation': 5000, 'test': 5000},
            'keypoints': len(vocabulary()['keypoint_order']),
            'features': len(vocabulary()['feature_names']),
            'defects': len(vocabulary()['defects']),
        },
        'note': 'Train once; every later cell loads this file. Not production Qwen3-4B/GGUF weights.',
    }
    payload = json.dumps(TRAINING_RESULTS, ensure_ascii=False, indent=2) + '\n'
    TRAINING_RESULTS_PATH.write_text(payload, encoding='utf-8')
    TRAINING_RESULTS_AR_PATH.write_text(payload, encoding='utf-8')
    print('✅ تم حفظ:', TRAINING_RESULTS_PATH.name, 'و', TRAINING_RESULTS_AR_PATH.name)

print(json.dumps({
    'qwen_base': TRAINING_RESULTS['qwen_adapter']['base_model'],
    'adapter': TRAINING_RESULTS['qwen_adapter']['adapter_dir'],
    'mlp_weights': TRAINING_RESULTS['stickman_mlp']['weights'],
    'mlp_error_px': TRAINING_RESULTS['stickman_mlp']['mlp_mean_joint_error_px'],
    'cpu_test': TRAINING_RESULTS['defect_detector']['cpu_final_test_score'],
}, ensure_ascii=False, indent=2))

## 2) إنشاء المحتوى من العنوان + الفكرة + النص المرجعي

هنا **يُستخدم ملف التدريب تلقائياً**: نحمّل Qwen بالadapter المحفوظ، ونعطيه عنوانك وفكرتك والنص الذي تريد الفيديو يتمحور حوله. هو يولّد `title`, و3–8 مقاطع، ولكل مقطع `narration` كامل + `caption` + `pose` + `camera`. النص المنطوق يُمرّر على معالج العربية لتحويل المصطلحات التقنية إلى قراءة عربية قبل TTS.

In [ ]:
from voice2reel.arabic_text import prepare_narration, spoken_form
from voice2reel.scriptwriter.generate import ScriptGenerator
from voice2reel.scriptwriter.schema import SYSTEM_PROMPT, parse_script

WRITER = ScriptGenerator(
    adapter=TRAINING_RESULTS['qwen_adapter']['adapter_dir'], device='cuda'
)
ACTIVE_REQUESTS = (
    VIDEO_REQUESTS if GENERATE_ALL_IDEAS else [VIDEO_REQUESTS[SELECTED_IDEA_INDEX]]
)
POSE_TO_GESTURE = {
    'IDLE': 'present', 'SHOCK': 'wave', 'CELEB': 'thumbs_up',
    'SUSPECT': 'think', 'LEAN': 'shrug', 'TYPE_A': 'point',
    'LOOK_L': 'present', 'LOOK_R': 'point',
}
CAMERA_TO_STICK = {
    'far': 'wide', 'medium': 'medium', 'close': 'close',
    'extreme': 'close', 'left': 'wide', 'right': 'medium',
}
POSE_TO_EMOTION = {
    'IDLE': 'neutral', 'SHOCK': 'surprised', 'CELEB': 'happy',
    'SUSPECT': 'concerned', 'LEAN': 'thinking', 'TYPE_A': 'neutral',
    'LOOK_L': 'neutral', 'LOOK_R': 'neutral',
}
VISUAL_CYCLE = ('none', 'list', 'flow', 'compare', 'code')


def estimate_seconds(text):
    words = max(1, len(text.split()))
    return round(max(2.5, min(8.0, words / 2.2)), 2)


def generate_script_for(request):
    grounding = (
        SYSTEM_PROMPT
        + '\nاستخدم النص المرجعي التالي كمصدر للمعنى فقط، ولا تخترع حقائق أو أرقاماً أو وعوداً خارجه. '
        + 'اجعل كل مقطع يخدم العنوان والفكرة، واكتب نصاً منطوقاً كاملاً مناسباً لكل لقطة.'
    )
    user = (
        f'عنوان الفيديو: {request["title"]}\n'
        f'الفكرة: {request["idea"]}\n'
        f'النص المرجعي: {request["source_text"]}\n'
        f'المدة المستهدفة: {DURATION} ثانية\nاللغة: {LANGUAGE}'
    )
    messages = [{'role': 'system', 'content': grounding}, {'role': 'user', 'content': user}]
    trace = []
    for attempt in range(1, 4):
        raw = WRITER.complete(messages, max_tokens=1200)
        trace.append({'attempt': attempt, 'raw_response': raw})
        try:
            script = parse_script(raw, DURATION, LANGUAGE)
            script = script.model_copy(update={'title': request['title']})
            return script, trace
        except ValueError as exc:
            trace[-1]['error'] = str(exc)
            messages.extend([
                {'role': 'assistant', 'content': raw},
                {'role': 'user', 'content': 'صحح JSON والتزم بالمخطط والمدة واللغة. الخطأ: ' + str(exc)[:700]},
            ])
    raise ValueError('Qwen لم يولّد سكربتاً صالحاً بعد 3 محاولات')


CONTENT_RUNS = []
for index, request in enumerate(ACTIVE_REQUESTS):
    run_dir = VIDEOS_DIR / f'video_{index:02d}'
    run_dir.mkdir(parents=True, exist_ok=True)
    script, trace = generate_script_for(request)
    (run_dir / 'script.json').write_text(
        json.dumps(script.model_dump(), ensure_ascii=False, indent=2) + '\n', encoding='utf-8'
    )
    (run_dir / 'script.trace.json').write_text(
        json.dumps(trace, ensure_ascii=False, indent=2) + '\n', encoding='utf-8'
    )
    (run_dir / 'narration.txt').write_text(script.narration() + '\n', encoding='utf-8')
    scene_specs = []
    for beat_index, beat in enumerate(script.beats):
        try:
            spoken = prepare_narration(beat.narration)
        except ValueError:
            spoken, _unresolved = spoken_form(beat.narration)
        visual = VISUAL_CYCLE[beat_index % len(VISUAL_CYCLE)]
        scene_specs.append({
            'index': beat_index,
            'caption': beat.caption,
            'display_narration': beat.narration,
            'spoken': spoken,
            'pose': beat.pose,
            'original_camera': beat.camera,
            'gesture': POSE_TO_GESTURE[beat.pose],
            'camera': CAMERA_TO_STICK[beat.camera],
            'emotion': POSE_TO_EMOTION[beat.pose],
            'visual': visual,
            'panel_visible': visual != 'none',
            'intent': 'hook' if beat_index == 0 else ('close' if beat_index == len(script.beats) - 1 else 'explain'),
            'words': len(beat.narration.split()),
            'estimated_seconds': estimate_seconds(spoken),
        })
    content = {
        'request': request, 'title': script.title, 'language': script.language,
        'target_duration_seconds': script.target_duration_seconds,
        'scene_specs': scene_specs,
        'training_results': str(TRAINING_RESULTS_PATH),
        'reused_training': True,
    }
    (run_dir / 'content.json').write_text(
        json.dumps(content, ensure_ascii=False, indent=2) + '\n', encoding='utf-8'
    )
    CONTENT_RUNS.append({'run_dir': run_dir, 'request': request, 'script': script,
                         'scene_specs': scene_specs, 'content': content})
    print('generated:', script.title, '| scenes:', len(scene_specs),
          '| words:', sum(s['words'] for s in scene_specs))

## 3) تحميل أوزان الرسم من نفس الملف + تجهيز الرندر

ما في تدريب جديد هنا. نقرأ `stickman_mlp.weights` من `model_training_results.json`، ونبني نفس MLP، ونستخدمه للتنبؤ بـ 18 نقطة لكل إطار.

In [ ]:
from voice2reel.stickman_artist import draw_frame
from scripts.build_stickman_drawing_data import raw_keypoints, place_keypoints

SAVED = torch.load(str(Path(TRAINING_RESULTS['stickman_mlp']['weights'])),
                    map_location=DEVICE, weights_only=False)
MLP_MODEL = nn.Sequential(
    nn.Linear(SAVED['input_dim'], 128), nn.ReLU(),
    nn.Linear(128, 128), nn.ReLU(),
    nn.Linear(128, SAVED['output_dim']),
).to(DEVICE)
MLP_MODEL.load_state_dict(SAVED['state_dict'])
MLP_MODEL.eval()
MEAN = np.asarray(SAVED['mean'], dtype=np.float32)
STD = np.asarray(SAVED['std'], dtype=np.float32)
DESIGNER = StickmanArtist()
VOCAB = vocabulary()
JOINT_GAP_INDEX = VOCAB['feature_names'].index('joint_gap')
BASE_BY_CAMERA = {
    camera: next(row for row in rows('validation')
                 if is_professional(row) and row['camera'] == camera)
    for camera in ('wide', 'medium', 'close')
}


def predict_points(row):
    design = ((DESIGNER.design(row) - MEAN) / STD).astype(np.float32)
    with torch.no_grad():
        output = MLP_MODEL(torch.from_numpy(design).unsqueeze(0).to(DEVICE)).cpu().numpy()
    points = np.clip(output.reshape(18, 2) * CANVAS, [0.0, 0.0], CANVAS)
    return [[round(float(a), 2), round(float(b), 2)] for a, b in points]


def render_frames(scene_specs, frames_dir, duration_key):
    frames_dir.mkdir(parents=True, exist_ok=True)
    for old in frames_dir.glob('frame_*.png'):
        old.unlink()
    previous_gesture = 'wave'
    frame_number = 0
    bounds_ok = True
    for scene in scene_specs:
        seconds = float(scene[duration_key])
        frames = max(1, int(round(seconds * FPS)))
        base = BASE_BY_CAMERA[scene['camera']]
        for frame_index in range(frames):
            phase = round(frame_index / max(1, frames - 1), 3)
            frame_row = dict(base)
            frame_row.update({
                'id': f"generated_{scene['index']:02d}_{frame_index:04d}",
                'split': 'generated', 'defect': None,
                'caption': scene['caption'],
                'gesture': scene['gesture'], 'previous_gesture': previous_gesture,
                'camera': scene['camera'], 'emotion': scene['emotion'],
                'intent': scene['intent'], 'visual': scene['visual'],
                'panel_visible': scene['panel_visible'],
                'phase': phase, 'features': list(base['features']),
                'style': dict(base['style']),
                'actor_box': list(base['actor_box']),
            })
            # Rebuild target geometry from current gesture/phase/camera rather than
            # borrowing stale geometry metadata from a validation example.
            target, target_scale = place_keypoints(
                raw_keypoints(scene['gesture'], previous_gesture, phase),
                scene['camera'], scene['panel_visible'],
            )
            frame_row['scale'] = target_scale
            frame_row['points'] = [target[key] for key in VOCAB['keypoint_order']]
            frame_row['actor_box'] = [
                min(p[0] for p in frame_row['points']), min(p[1] for p in frame_row['points']),
                max(p[0] for p in frame_row['points']), max(p[1] for p in frame_row['points']),
            ]
            frame_row['features'][JOINT_GAP_INDEX] = 0.0
            predicted = predict_points(frame_row)
            # The learned regressor is only used when it improves geometry enough;
            # otherwise preserve the connected, bounded procedural IK reference.
            error = float(np.mean(np.linalg.norm(
                np.asarray(predicted) - np.asarray(frame_row['points']), axis=1)))
            chosen = predicted if error <= 6.0 else frame_row['points']
            bounds_ok &= all(0.0 <= x <= 540.0 and 0.0 <= y <= 960.0 for x, y in chosen)
            draw_frame(frame_row, points=chosen,
                       output=frames_dir / f'frame_{frame_number:05d}.png')
            frame_number += 1
        previous_gesture = scene['gesture']
    return frame_number, bool(bounds_ok)

print('MLP loaded from training results | input:', SAVED['input_dim'], '| output:', SAVED['output_dim'])

## 4) إنشاء الفيديو بدون صوت أولاً

نستخدم التوقيتات المُقدّرة من عدد الكلمات، وننتج `silent_video.mp4` بمسار صوت صامت. هذه مرحلة مستقلة قبل أي TTS، زي ما طلبت.

In [ ]:
def write_silent_video(frames_dir, frame_count, output_path):
    command = [
        FFMPEG, '-y', '-loglevel', 'error',
        '-framerate', str(FPS), '-i', str(frames_dir / 'frame_%05d.png'),
        '-f', 'lavfi', '-i', 'anullsrc=channel_layout=stereo:sample_rate=44100',
        '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-r', str(FPS),
        '-c:a', 'aac', '-shortest', '-movflags', '+faststart', str(output_path),
    ]
    subprocess.run(command, check=True)
    return round(frame_count / FPS, 2)

for run in CONTENT_RUNS:
    run_dir = run['run_dir']
    frames_dir = run_dir / 'silent_frames'
    silent_path = run_dir / 'silent_video.mp4'
    frame_count, bounds_ok = render_frames(run['scene_specs'], frames_dir, 'estimated_seconds')
    duration = write_silent_video(frames_dir, frame_count, silent_path)
    report = {
        'stage': 'silent_video', 'frames': frame_count, 'fps': FPS,
        'estimated_duration_seconds': duration,
        'scene_durations': [s['estimated_seconds'] for s in run['scene_specs']],
        'predicted_joints_inside_canvas': bounds_ok,
        'training_results_used': str(TRAINING_RESULTS_PATH),
    }
    (run_dir / 'silent_report.json').write_text(
        json.dumps(report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8'
    )
    run['silent_video'] = silent_path
    run['silent_frames'] = frames_dir
    run['silent_report'] = report
    print('silent:', silent_path.name, '| frames:', frame_count, '| seconds:', duration)

## 5) إنشاء الصوت بعد الفيديو الصامت

لكل مشهد نولّد MP3 من `spoken` الذي كتبه Qwen. نقيس مدة كل ملف بـ ffmpeg، ونجمع الملفات في `narration.mp3`. إن فشل `edge-tts` أو جعلته `False`، يرجع تلقائياً لصوت صامت بنفس المدة المُقدّرة.

In [ ]:
import asyncio
import edge_tts
import nest_asyncio

nest_asyncio.apply()


def ffmpeg_duration(path):
    probe = subprocess.run(
        [FFMPEG, '-i', str(path), '-f', 'null', '-'],
        capture_output=True, text=True,
    )
    match = re.search(r'Duration: (\d+):(\d+):(\d+\.\d+)', probe.stderr)
    if not match:
        raise RuntimeError(f'Cannot read audio duration: {path}')
    hours, minutes, seconds = match.groups()
    return round(int(hours) * 3600 + int(minutes) * 60 + float(seconds), 3)


def silent_audio(path, seconds):
    subprocess.run([
        FFMPEG, '-y', '-loglevel', 'error',
        '-f', 'lavfi', '-i', 'anullsrc=channel_layout=stereo:sample_rate=44100',
        '-t', f'{seconds:.3f}', '-c:a', 'libmp3lame', str(path),
    ], check=True)


async def synthesize_all():
    for run in CONTENT_RUNS:
        audio_dir = run['run_dir'] / 'audio'
        audio_dir.mkdir(parents=True, exist_ok=True)
        for scene in run['scene_specs']:
            target = audio_dir / f"scene_{scene['index']:02d}.mp3"
            if USE_EDGE_TTS:
                try:
                    communicator = edge_tts.Communicate(scene['spoken'], VOICE)
                    await communicator.save(str(target))
                except Exception as exc:
                    print('TTS fallback for scene', scene['index'], repr(exc))
                    raise RuntimeError(
                        f"Voice synthesis failed for scene {scene['index']}; "
                        "set USE_EDGE_TTS=False explicitly for a silent preview"
                    ) from exc
            else:
                silent_audio(target, scene['estimated_seconds'])
            scene['audio_path'] = str(target)
            scene['audio_duration'] = ffmpeg_duration(target)


asyncio.run(synthesize_all())

for run in CONTENT_RUNS:
    audio_dir = run['run_dir'] / 'audio'
    concat_file = audio_dir / 'concat.txt'
    concat_file.write_text(
        '\n'.join(f"file '{scene['audio_path']}'" for scene in run['scene_specs']) + '\n',
        encoding='utf-8',
    )
    narration = run['run_dir'] / 'narration.mp3'
    subprocess.run([
        FFMPEG, '-y', '-loglevel', 'error', '-f', 'concat', '-safe', '0',
        '-i', str(concat_file), '-c:a', 'libmp3lame', str(narration),
    ], check=True)
    audio_report = {
        'voice': VOICE if USE_EDGE_TTS else 'silent',
        'per_scene_seconds': [s['audio_duration'] for s in run['scene_specs']],
        'total_seconds': round(sum(s['audio_duration'] for s in run['scene_specs']), 3),
        'concatenated_duration': ffmpeg_duration(narration),
        'narration_path': str(narration),
    }
    (run['run_dir'] / 'audio_report.json').write_text(
        json.dumps(audio_report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8'
    )
    run['narration'] = narration
    run['audio_report'] = audio_report
    print('audio:', narration.name, '| seconds:', audio_report['concatenated_duration'])

## 6) تركيب الصوت والفيديو «بطريقة مضبوطة»

نعيد رسم الإطارات باستخدام **المدة الحقيقية لكل مقطع صوتي** (مش التقديرية)، ثم ندمج `narration.mp3` مع الإطارات. بهذا الكلام والترجمة وحركة الـ Stickman متزامنة.

In [ ]:
for run in CONTENT_RUNS:
    run_dir = run['run_dir']
    final_frames = run_dir / 'final_frames'
    final_video = run_dir / 'final_video.mp4'
    frame_count, bounds_ok = render_frames(run['scene_specs'], final_frames, 'audio_duration')
    command = [
        FFMPEG, '-y', '-loglevel', 'error',
        '-framerate', str(FPS), '-i', str(final_frames / 'frame_%05d.png'),
        '-i', str(run['narration']),
        '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-r', str(FPS),
        '-c:a', 'aac', '-shortest', '-movflags', '+faststart', str(final_video),
    ]
    subprocess.run(command, check=True)
    production_report = {
        'title': run['content']['title'],
        'request': run['request'],
        'training_results': str(TRAINING_RESULTS_PATH),
        'reused_training_without_retraining': True,
        'qwen_adapter': TRAINING_RESULTS['qwen_adapter']['adapter_dir'],
        'stickman_mlp_weights': TRAINING_RESULTS['stickman_mlp']['weights'],
        'content_json': str(run_dir / 'content.json'),
        'script_json': str(run_dir / 'script.json'),
        'silent_video': str(run['silent_video']),
        'narration': str(run['narration']),
        'final_video': str(final_video),
        'frames': frame_count,
        'fps': FPS,
        'video_seconds': round(frame_count / FPS, 2),
        'audio_seconds': run['audio_report']['concatenated_duration'],
        'voice': run['audio_report']['voice'],
        'scene_timing_source': 'measured audio durations',
        'predicted_joints_inside_canvas': bounds_ok,
        'limitations': [
            'Demo pipeline, not production Qwen3-4B/GGUF or a neural image generator.',
            'edge-tts is an external service and is not cleared for commercial use here.',
            'No human editorial/aesthetic review was performed.',
        ],
    }
    (run_dir / 'production_report.json').write_text(
        json.dumps(production_report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8'
    )
    run['final_video'] = final_video
    run['production_report'] = production_report
    print('final:', final_video.name, '| frames:', frame_count,
          '| video:', production_report['video_seconds'],
          '| audio:', production_report['audio_seconds'])

## 7) العرض والتنزيل

يعرض الفيديو النهائي، وينزّل: الفيديو، الصوت، السكربت/المحتوى، تقرير الإنتاج، و**ملف نتائج التدريب** نفسه لو بدك تنقله لجهاز/جلسة ثانية وتستخدمه بدون إعادة تدريب.

In [ ]:
from IPython.display import Video, display

for run in CONTENT_RUNS:
    print('▶', run['content']['title'])
    display(Video(str(run['final_video']), embed=True))

try:
    from google.colab import files
    files.download(str(TRAINING_RESULTS_PATH))
    print('Downloaded training results:', TRAINING_RESULTS_PATH.name)
    for run in CONTENT_RUNS:
        for artifact in (
            run['final_video'], run['narration'],
            run['run_dir'] / 'content.json', run['run_dir'] / 'script.json',
            run['run_dir'] / 'production_report.json',
        ):
            files.download(str(artifact))
    print('Downloaded all video artifacts.')
except ImportError:
    print('Not running in Colab; artifacts are in', WORK)

## الخلاصة بصراحة

- **اه**: صار فيه ملف `model_training_results.json` / `نتائج_تدريب_الموديل.json` يُكتب بعد التدريب، والخلايا التالية تقرأه تلقائياً. ما بترجع تدرّب كل فيديو.
- العنوان + الفكرة + النص المرجعي → Qwen بالadapter المحفوظ يولّد المحتوى الكامل لكل مشهد.
- بعدها: فيديو صامت → صوت → دمج متزامن مع إعادة ضبط التوقيت.
- تغيّر `VIDEO_REQUESTS` أو `SELECTED_IDEA_INDEX` وتعيد التشغيل من **خلية المحتوى** فقط؛ التدريب ما بيتكرر.
- لو بدك تدريب أعمق: زِد `QWEN_MAX_STEPS` / `QWEN_EPOCHS` / `MLP_EPOCHS` واعمل `FORCE_RETRAIN = True` مرة واحدة، ثم أرجعه `False`.
- `notebooks/05_quality_loop.ipynb` لسه مطلوب لتقييم كاتب الإنتاج على T4 و`final_report.json`؛ هذا الدفتر استوديو إنتاج تجريبي للـ Stickman، مش بديل عنه.